In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
np.random.seed(0)
torch.manual_seed(0)

class MLP(nn.Module):  
    def __init__(self, layers):
        super().__init__()
        net = []
        for i in range(len(layers) - 2):
            net.append(nn.Linear(layers[i], layers[i+1]))
            net.append(nn.Tanh())
        net.append(nn.Linear(layers[-2], layers[-1]))
        self.net = nn.Sequential(*net)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)



class NNs:
    def __init__(self, x, y, u, v, p, layers, lr=1e-4):
        X = np.concatenate([x, y], axis=1).astype(np.float32)

        self.lb = torch.tensor(X.min(0), dtype=torch.float32, device=device)
        self.ub = torch.tensor(X.max(0), dtype=torch.float32, device=device)


        self.x = torch.tensor(X[:, 0:1], dtype=torch.float32, device=device)
        self.y = torch.tensor(X[:, 1:2], dtype=torch.float32, device=device)
        
        self.u = torch.tensor(u.astype(np.float32), dtype=torch.float32, device=device)
        self.v = torch.tensor(v.astype(np.float32), dtype=torch.float32, device=device)
        self.p = torch.tensor(p.astype(np.float32), dtype=torch.float32, device=device)

        self.u_mean = self.u.mean()
        self.u_std  = self.u.std() + 1e-8
        self.v_mean = self.v.mean()
        self.v_std  = self.v.std() + 1e-8      
        self.p_mean = self.p.mean()
        self.p_std  = self.p.std() + 1e-8
        
        self.u_n = (self.u - self.u_mean) / self.u_std
        self.v_n = (self.v - self.v_mean) / self.v_std
        self.p_n = (self.p - self.p_mean) / self.p_std

        
        self.model = MLP(layers).to(device)
        self.opt = torch.optim.Adam(self.model.parameters(), lr=lr, weight_decay=1e-8)
        self.mse = nn.MSELoss()
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.opt, mode="min", factor=0.5, patience=2000
        )

    
    def neural_net(self, X):
        eps = 1e-8
        H = 2.0 * (X - self.lb)/(self.ub - self.lb + eps) - 1.0
        return self.model(H)

    def net_U(self, x, y):
        X = torch.cat([x, y], dim=1)
        Y = self.neural_net(X)
        u_pred_n =  Y[:, 0:1]
        v_pred_n  = Y[:, 1:2]
        p_pred_n  = Y[:, 2:3]
        return  u_pred_n, v_pred_n, p_pred_n
        
    def net_diff(self, x, y,diff):
        x = x.clone().detach().requires_grad_(True)
        y = y.clone().detach().requires_grad_(True)

        u_n, v_n, p_n = self.net_U(x, y)

        u_x_n = torch.autograd.grad(u_n, x, grad_outputs=torch.ones_like(u_n),create_graph=True)[0]
        u_y_n = torch.autograd.grad(u_n, y, grad_outputs=torch.ones_like(u_n),create_graph=True)[0]
        v_x_n = torch.autograd.grad(v_n, x, grad_outputs=torch.ones_like(v_n),create_graph=True)[0]
        v_y_n = torch.autograd.grad(v_n, y, grad_outputs=torch.ones_like(v_n),create_graph=True)[0]
        p_x_n = torch.autograd.grad(p_n, x, grad_outputs=torch.ones_like(p_n),create_graph=True)[0]
        p_y_n = torch.autograd.grad(p_n, y, grad_outputs=torch.ones_like(p_n),create_graph=True)[0]

        if diff == 2:
            u_xx_n = torch.autograd.grad(u_x_n, x, grad_outputs=torch.ones_like(u_x_n), create_graph=True)[0]
            u_yy_n = torch.autograd.grad(u_y_n, y, grad_outputs=torch.ones_like(u_y_n), create_graph=True)[0]

            v_xx_n = torch.autograd.grad(v_x_n, x, grad_outputs=torch.ones_like(v_x_n), create_graph=True)[0]
            v_yy_n = torch.autograd.grad(v_y_n, y, grad_outputs=torch.ones_like(v_y_n), create_graph=True)[0]
            
            return (u_x_n, u_y_n, u_xx_n, u_yy_n), (v_x_n, v_y_n, v_xx_n, v_yy_n), (p_x_n, p_y_n)
        return (u_x_n, u_y_n), (v_x_n, v_y_n), (p_x_n, p_y_n)
    
    def train(self, nIter, clip_grad=1.0):
        print("Start training...")
        start_time = time.time()

        for it in range(nIter):
            self.opt.zero_grad()
            
            u_pred_n, v_pred_n, p_pred_n = self.net_U(self.x, self.y)

            loss = (self.mse(self.u_n, u_pred_n) +
                    self.mse(self.v_n, v_pred_n) +
                    self.mse(self.p_n, p_pred_n)
                   )
            loss.backward()
            if clip_grad is not None:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=clip_grad)

            self.opt.step()
            self.scheduler.step(loss.item())

            if it % 200 == 0 or it == nIter-1:
                elapsed = time.time() - start_time
                lr_now = self.opt.param_groups[0]["lr"]
                print(f"Iter: {it:5d}, Loss: {loss.item():.3e}, lr: {lr_now:.1e}, Time: {elapsed:.2f}s")
                start_time = time.time()

    def predict(self, x_star, y_star):
        x_star = torch.as_tensor(x_star, dtype=torch.float32, device=device)
        y_star = torch.as_tensor(y_star, dtype=torch.float32, device=device)
    
        with torch.no_grad():
            u_pred_n, v_pred_n, p_pred_n = self.net_U(x_star, y_star)    
            
        (u_x_n, u_y_n, u_xx_n, u_yy_n), (v_x_n, v_y_n, v_xx_n, v_yy_n), (p_x_n, p_y_n)= self.net_diff(x_star, y_star,diff=2)

        u_pred = u_pred_n * self.u_std + self.u_mean
        v_pred = v_pred_n * self.v_std + self.v_mean
        p_pred = p_pred_n * self.p_std + self.p_mean


        u_x = u_x_n * self.u_std
        u_y = u_y_n * self.u_std
        v_x = v_x_n * self.v_std
        v_y = v_y_n * self.v_std       
        p_x = p_x_n * self.p_std
        p_y = p_y_n * self.p_std

        u_xx = u_xx_n * self.u_std
        u_yy = u_yy_n * self.u_std
    
        v_xx = v_xx_n * self.v_std
        v_yy = v_yy_n * self.v_std
        
        return (
            u_pred.detach().cpu().numpy(), (u_x.detach().cpu().numpy(), u_y.detach().cpu().numpy()), 
            (u_xx.detach().cpu().numpy(), u_yy.detach().cpu().numpy()),
            
            v_pred.detach().cpu().numpy(), (v_x.detach().cpu().numpy(), v_y.detach().cpu().numpy()), 
            (v_xx.detach().cpu().numpy(), v_yy.detach().cpu().numpy()),
            
            p_pred.detach().cpu().numpy(), (p_x.detach().cpu().numpy(), p_y.detach().cpu().numpy()),          
        )



if __name__ == "__main__":
    layers = [2, 80, 80, 80, 80, 80, 80, 80, 3]
    # x,y,rho,u,v,p,u_t,v_t
    train_data = np.loadtxt('Taylor_Green_vortex_flow_LBM.txt',skiprows=1).astype(np.float32)
    x   = train_data[:, 0:1]
    y   = train_data[:, 1:2]
    rho = train_data[:, 2:3]
    u   = train_data[:, 3:4]
    v   = train_data[:, 4:5]
    p   = train_data[:, 5:6]
    u_t = train_data[:, 6:7]
    v_t = train_data[:, 7:8]

    print("Raw output scales:")
    print("u min/max/mean/std:", u.min(), u.max(), u.mean(), u.std())
    print("v min/max/mean/std:", v.min(), v.max(), v.mean(), v.std())
    print("p min/max/mean/std:", p.min(), p.max(), p.mean(), p.std())
    model = NNs(x, y, u, v, p, layers, lr=1e-5)
    model.train(200000, clip_grad=1.0)

    
    (u_pred, (u_x, u_y), (u_xx, u_yy), 
     v_pred, (v_x, v_y), (v_xx, v_yy), 
     p_pred, (p_x, p_y) 
    ) = model.predict(x, y)
    
    u_pred = np.array(u_pred.tolist(), dtype=np.float32)
    v_pred = np.array(v_pred.tolist(), dtype=np.float32)
    p_pred = np.array(p_pred.tolist(), dtype=np.float32)
    
    error_u = np.linalg.norm(u - u_pred, 2) / np.linalg.norm(u, 2)
    error_v = np.linalg.norm(v - v_pred, 2) / np.linalg.norm(v, 2)
    error_p = np.linalg.norm(p - p_pred, 2) / np.linalg.norm(p, 2)
    print("\n Final error:")
    print(f"u_error: {error_u:.3e}")
    print(f"v_error: {error_v:.3e}")
    print(f"p_error: {error_p:.3e}")


    row = x.shape[0]
    filename = "Taylor_Green_vortex.dat"
    with open(filename, "w") as f:
        f.write("x,y,u,v,p,rho,u_x,u_y,v_x,v_y,p_x,p_y,u_xx,u_yy,v_xx,v_yy,u_t,v_t\n")
    
        for i in range(row):
            f.write(
                f"{x[i, 0]:.6e} {y[i, 0]:.6e} "
                f"{u[i, 0]:.6e} {v[i, 0]:.6e} {p[i, 0]:.6e} {rho[i, 0]:.6e} "
                f"{u_x[i, 0]:.6e} {u_y[i, 0]:.6e} {v_x[i, 0]:.6e} {v_y[i, 0]:.6e} "
                f"{p_x[i, 0]:.6e} {p_y[i, 0]:.6e} "
                f"{u_xx[i, 0]:.6e} {u_yy[i, 0]:.6e} {v_xx[i, 0]:.6e} {v_yy[i, 0]:.6e} "
                f"{u_t[i, 0]:.6e} {v_t[i, 0]:.6e}\n"
            )
    
    print(f"The data has been written to the file: {filename}")    